# Projeto A3 — Comitê de Classificadores (NASA JPL SBDB)

Universidade São Judas Tadeu — Inteligência Artificial

## Membros:
| Nome | RA |
|---|---|
| Matheus Santos Morais Arruda | 825218417 |
| Lucca Campello Rodrigues dos Santos | 82525684 |
| Jorge Antonio de Paula Tosi  | 825220685 |
| Guilherme Caposse Tabler | 825126059 |
| Matheus Miano | 

## Visão geral do notebook

- **Base de dados:** `sbdb_asteroides.csv` (NASA Jet Propulsion Laboratory — Small-Body Database)
- **Variável-alvo:** `pha` (*Potentially Hazardous Asteroid*), valores `Y` / `N`
- **Objetivo:** prever se um asteroide é **potencialmente perigoso** para a Terra a partir de parâmetros orbitais.

### O que esse notebook entrega (mapeado para o enunciado A3)

| Seção do enunciado | Onde está no notebook |
|---|---|
| 3.3 Pré-processamento | Tópicos 2 e 3 |
| 3.4 ≥ 3 classificadores individuais | Tópico 5 (5 modelos) |
| 3.5 Comitê (ensemble) | Tópico 6 (Soft + Hard Voting) |
| 3.6 Avaliação (Accuracy, Precision, Recall, F1, Matriz de Confusão) | Tópicos 5, 6 e 7 |

O notebook está dividido nos **mesmos tópicos numerados** que aparecem nos `print` do script `projeto_a3_comite_classificadores.py`, de modo que cada seção daqui corresponde a um bloco de log do script.

## 0) Importações e configuração global

Antes de tudo, importamos as bibliotecas. Cada uma cumpre um papel específico:

- **`warnings`** → silencia avisos do scikit-learn (ex.: convergência do MLP) para o log ficar limpo.
- **`numpy` / `pandas`** → manipulação numérica e tabular.
- **`matplotlib.pyplot`** → geração dos gráficos (barras de métricas e matrizes de confusão).
- **`sklearn.model_selection`** → `train_test_split`, validação cruzada estratificada (`StratifiedKFold`) e `cross_val_score`.
- **`sklearn.preprocessing.StandardScaler`** → padroniza as features (média 0, desvio 1). Essencial para KNN, SVM e MLP.
- **`sklearn.impute.SimpleImputer`** → preenche valores ausentes; aqui usamos a mediana (robusta a outliers).
- **Classificadores individuais:** `DecisionTreeClassifier`, `KNeighborsClassifier`, `GaussianNB`, `SVC`, `MLPClassifier`.
- **`VotingClassifier`** → implementa o **comitê** (combina os modelos em uma única predição).
- **`sklearn.metrics`** → as métricas exigidas pelo enunciado: accuracy, precision, recall, F1, matriz de confusão e `classification_report`.

`RANDOM_STATE = 42` fixa a aleatoriedade — todo split/sampling/inicialização passa a ser **reprodutível**.

In [1]:
import warnings
warnings.filterwarnings("ignore")  # esconde warnings (ex.: ConvergenceWarning do MLP)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Utilitários de avaliação/validação
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler        # padronização (z-score)
from sklearn.impute import SimpleImputer                # imputação de NaN

# Os 5 classificadores individuais
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

# Comitê (ensemble de votação)
from sklearn.ensemble import VotingClassifier

# Métricas exigidas pelo enunciado A3 (item 3.6)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

RANDOM_STATE = 42                                              # semente global
DATA_PATH = "../projeto_a3/data/raw/sbdb_asteroides.csv"      # caminho relativo ao notebook

## 1) Carregamento da base + EDA básica

Lemos o CSV bruto do SBDB e olhamos rapidamente:

- **`df.shape`** → quantas linhas (asteroides) × colunas (atributos). Mostra a escala do dataset.
- **`df["pha"].value_counts(dropna=False)`** → distribuição do alvo, contando inclusive os `NaN`.

Esse `value_counts` já revela o principal desafio do problema: o dataset é **extremamente desbalanceado** — a classe `Y` (perigoso) representa cerca de **0,18%** do total. Isso justifica a estratégia de balanceamento aplicada no tópico 3.

In [ ]:
print("=" * 60)
print("1) CARREGAMENTO DA BASE")
print("=" * 60)

# low_memory=False evita o warning de tipos mistos por chunk
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Shape original: {df.shape}")
print("\nDistribuicao do target 'pha' (antes do pre-processamento):")
print(df["pha"].value_counts(dropna=False))

## 2) Pré-processamento

Quatro passos clássicos antes de treinar qualquer modelo:

1. **Remover linhas sem rótulo** (`pha` ausente) — não dá para treinar nem avaliar com `y = NaN`.
2. **Codificar o alvo** em binário: `Y → 1` (perigoso), `N → 0` (não perigoso).
3. **Selecionar features** estritamente *orbitais* + magnitude absoluta:
   - `q` (periélio), `ad` (afélio), `H` (magnitude absoluta — proxy de tamanho),
   - `ma` (anomalia média), `per` (período orbital), `e` (excentricidade),
   - `a` (semieixo maior), `i` (inclinação orbital).
   
   ⚠️ **Por que NÃO usamos `moid` nem `neo`?** Porque a própria *definição oficial* de PHA depende deles — usá-los seria **data leakage** (a feature carrega a resposta) e inflaria artificialmente as métricas.
4. **Imputação** dos `NaN` com a **mediana** de cada coluna (robusta a outliers — e parâmetros orbitais costumam ter cauda longa).

In [ ]:
print("\n" + "=" * 60)
print("2) PRE-PROCESSAMENTO")
print("=" * 60)

# 2.1 Remove linhas sem rotulo (target nao pode ser NaN) e reseta indice
df = df.dropna(subset=["pha"]).reset_index(drop=True).copy()

# 2.2 Codifica o target: Y -> 1 (perigoso), N -> 0 (nao perigoso)
df["target"] = (df["pha"] == "Y").astype(int)

# 2.3 Selecao de features orbitais (sem moid/neo => evita data leakage)
feature_cols = ["q", "ad", "H", "ma", "per", "e", "a", "i"]
X = df[feature_cols].copy()
y = df["target"].copy()

# 2.4 Imputacao de NaN pela mediana (robusta a outliers)
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

print(f"Features usadas : {feature_cols}")
print(f"X.shape = {X.shape} | y.shape = {y.shape}")
print(f"Positivos (PHA=Y): {int(y.sum())} | Negativos (PHA=N): {int((y==0).sum())}")

## 3) Balanceamento — *undersampling* 1:3

**Problema:** a classe positiva (`PHA = Y`) é ~0,18% do dataset. Se treinarmos direto, qualquer modelo aprende a "chutar tudo N" e atinge ~99,8% de **acurácia inútil** — recall da classe positiva próximo de zero.

**Estratégia escolhida:** *undersampling* da classe majoritária.

- Mantemos **todos** os asteroides perigosos (positivos).
- Amostramos aleatoriamente `3 × (nº de positivos)` negativos.
- Proporção final → **1 perigoso : 3 não-perigosos**.

Por que 1:3 e não 1:1? 1:1 jogaria fora dados demais e tornaria o modelo otimista demais (perde a noção de quão rara é a classe positiva no mundo real). 1:3 é um meio-termo bom para datasets fortemente desbalanceados.

In [ ]:
print("\n" + "=" * 60)
print("3) BALANCEAMENTO (undersampling 1:3)")
print("=" * 60)

# Indices dos positivos (todos) e dos negativos amostrados (3x os positivos)
pos_idx = y[y == 1].index
neg_idx = y[y == 0].sample(n=len(pos_idx) * 3, random_state=RANDOM_STATE).index
keep_idx = pos_idx.union(neg_idx)

# Subconjunto balanceado
X_bal = X.loc[keep_idx].reset_index(drop=True)
y_bal = y.loc[keep_idx].reset_index(drop=True)

print(f"Apos balanceamento: {X_bal.shape}")
print(y_bal.value_counts().rename({0: "Nao-Perigoso (0)", 1: "Perigoso (1)"}))

## 4) Split treino / teste + padronização

- **`train_test_split` com `stratify=y_bal`** → garante que a proporção 1:3 seja preservada **tanto no treino quanto no teste** (importante porque a classe positiva é a que mais nos interessa).
- **`test_size=0.25`** → 75% para treinar, 25% para avaliar.
- **`StandardScaler`** → centraliza (média = 0) e escala (desvio = 1) cada feature.
  - **Obrigatório para:** KNN (distâncias euclidianas), SVM com kernel RBF (sensibilidade à escala), MLP (saturação de ativações).
  - **Indiferente para:** Decision Tree (splits por threshold) e Gaussian NB (cada feature é modelada independentemente).
  - Aplicar para todos não atrapalha os dois últimos e **simplifica o pipeline** — usamos a mesma matriz `X_train_s` em todos os modelos.
- **Ponto crítico:** o `scaler` é *ajustado* (`fit`) apenas com o treino, e depois *aplicado* (`transform`) no teste — assim o teste continua "invisível" no momento do ajuste, sem vazamento estatístico.

In [ ]:
# Split estratificado 75/25
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.25,
    stratify=y_bal,                # mantem proporcao 1:3 em treino e teste
    random_state=RANDOM_STATE,
)

# Padronizacao (z-score). Fit so no treino para evitar vazamento.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"\nTreino: {X_train_s.shape} | Teste: {X_test_s.shape}")

## 5) Classificadores individuais (5 modelos)

O enunciado pede ≥ 3 classificadores. Usamos **5**, escolhidos por terem *vieses indutivos diferentes* (o que ajuda no ensemble, pois erros tendem a se compensar):

| Modelo | Como decide | Por que está aqui |
|---|---|---|
| **KNN (k=5)** | maioria entre os 5 vizinhos mais próximos | baseline geométrico, sensível à escala |
| **Gaussian Naive Bayes** | probabilidade assumindo features gaussianas e independentes | rápido, bom baseline probabilístico |
| **Decision Tree (max_depth=8)** | partições binárias do espaço | interpretável; `max_depth` controla overfitting |
| **SVM (kernel RBF)** | hiperplano em espaço transformado; `probability=True` habilita `predict_proba` (necessário p/ soft voting) | bom em fronteiras não-lineares |
| **MLP (64→32 neurônios)** | rede neural feed-forward | aprende interações não-lineares mais ricas |

Para cada modelo, treinamos e avaliamos no conjunto de teste, guardando **Accuracy / Precision / Recall / F1 / Matriz de Confusão** num dicionário `resultados` que será usado nos tópicos 6 e 7.

O **`classification_report`** mostra precision/recall/F1 *por classe*, o que é mais informativo do que só ver médias agregadas.

In [ ]:
print("\n" + "=" * 60)
print("5) TREINAMENTO DOS CLASSIFICADORES INDIVIDUAIS")
print("=" * 60)

# Dicionario {nome: instancia} - facilita iteracao + reuso no VotingClassifier
modelos = {
    "KNN":           KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":   GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=RANDOM_STATE),
    # probability=True e essencial para o soft voting funcionar com SVM
    "SVM":           SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "MLP":           MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=400,
                                   random_state=RANDOM_STATE),
}

# Armazena metricas de cada modelo para comparar depois
resultados = {}
for nome, mdl in modelos.items():
    mdl.fit(X_train_s, y_train)
    y_pred = mdl.predict(X_test_s)
    resultados[nome] = {
        "acc":  accuracy_score(y_test, y_pred),
        "prec": precision_score(y_test, y_pred),
        "rec":  recall_score(y_test, y_pred),
        "f1":   f1_score(y_test, y_pred),
        "cm":   confusion_matrix(y_test, y_pred),
    }
    print(f"\n--- {nome} ---")
    print(classification_report(y_test, y_pred,
                                target_names=["Nao-Perigoso", "Perigoso"]))

## 6) Comitê de classificadores (Voting Classifier)

Aqui está o coração do item 3.5 do enunciado: combinar os 5 modelos em um único **comitê**.

### Soft voting (estratégia principal)
Cada modelo retorna a **probabilidade** de cada classe via `predict_proba`. O comitê faz a **média** dessas probabilidades e decide pela classe de maior média.

- Aproveita a **confiança** de cada modelo — se um classificador está em cima do muro (`prob ≈ 0.51`), ele pesa menos no resultado.
- Mais estável que hard voting quando os modelos têm performance **heterogênea** (que é o nosso caso: Naive Bayes ≠ MLP em termos de viés).

### Hard voting (comparativo)
Cada modelo "vota" no rótulo final e a classe mais votada vence (maioria simples). É mais simples, mas perde a noção de confiança.

Treinamos os dois e adicionamos seus resultados ao mesmo dicionário `resultados`, para comparar tudo lado a lado no tópico 7.

In [ ]:
print("\n" + "=" * 60)
print("6) CONSTRUCAO DO COMITE (VOTING CLASSIFIER)")
print("=" * 60)

# VotingClassifier exige lista de tuplas (nome, estimador)
estimators = list(modelos.items())

# --- Soft Voting: media das probabilidades ---
ensemble_soft = VotingClassifier(estimators=estimators, voting="soft")
ensemble_soft.fit(X_train_s, y_train)
y_pred_soft = ensemble_soft.predict(X_test_s)

# --- Hard Voting: maioria de votos ---
ensemble_hard = VotingClassifier(estimators=estimators, voting="hard")
ensemble_hard.fit(X_train_s, y_train)
y_pred_hard = ensemble_hard.predict(X_test_s)

# Mesmas metricas, agora para os dois comites
for nome, y_pred in [("Comite (Soft Voting)", y_pred_soft),
                     ("Comite (Hard Voting)", y_pred_hard)]:
    resultados[nome] = {
        "acc":  accuracy_score(y_test, y_pred),
        "prec": precision_score(y_test, y_pred),
        "rec":  recall_score(y_test, y_pred),
        "f1":   f1_score(y_test, y_pred),
        "cm":   confusion_matrix(y_test, y_pred),
    }
    print(f"\n--- {nome} ---")
    print(classification_report(y_test, y_pred,
                                target_names=["Nao-Perigoso", "Perigoso"]))

## 7) Resumo comparativo + visualizações

Agora consolidamos tudo. Construímos um `DataFrame` `df_metricas` em que **cada linha é um modelo** (5 individuais + 2 comitês) e **cada coluna é uma métrica** (Accuracy, Precision, Recall, F1).

Esse DataFrame é o insumo dos dois gráficos a seguir.

In [ ]:
print("\n" + "=" * 60)
print("7) RESUMO COMPARATIVO")
print("=" * 60)

# Constroi DataFrame: modelos nas linhas, metricas nas colunas (cm fica de fora)
df_metricas = pd.DataFrame(
    {n: {k: v for k, v in r.items() if k != "cm"} for n, r in resultados.items()}
).T.rename(columns={"acc": "Accuracy", "prec": "Precision",
                    "rec": "Recall", "f1": "F1-Score"})

print(df_metricas.round(4).to_string())

### 7.1) Gráfico de barras — `comparacao_metricas.png`

**O que este gráfico mostra:** uma comparação **lado a lado** das 4 métricas (Accuracy / Precision / Recall / F1) para cada um dos 7 modelos (5 individuais + 2 comitês).

- **Eixo X:** nome dos modelos (KNN, Naive Bayes, Decision Tree, SVM, MLP, Comitê Soft, Comitê Hard).
- **Eixo Y:** valor da métrica, de 0 a 1.05 (1.05 só para dar respiro visual no topo).
- **Cada cor representa uma métrica diferente** (colormap `viridis`).
- **`edgecolor="black"`** delimita cada barra → melhor leitura quando barras vizinhas são parecidas.
- **`xticks(rotation=20)`** evita que nomes longos se sobreponham.

**Como ler o gráfico:**
- Barras altas e equilibradas nas 4 cores → modelo balanceado.
- Recall alto + Precision baixo → modelo "chuta perigoso demais" (muitos falsos positivos).
- Recall baixo + Precision alto → modelo "acerta quando aposta, mas perde muitos perigosos" (problema sério neste domínio — não detectar um asteroide perigoso é o pior erro possível).
- Olhar especificamente as barras dos **dois comitês** → eles devem aparecer pelo menos comparáveis ao melhor individual; se forem **mais estáveis** entre as 4 métricas, o ensemble cumpriu seu papel.

In [ ]:
# 7.1 Grafico de barras: 4 metricas x 7 modelos
ax = df_metricas.plot(
    kind="bar",
    figsize=(12, 6),
    colormap="viridis",       # uma cor por metrica
    edgecolor="black",        # contorno deixa barras vizinhas distinguiveis
)
ax.set_title("Comparacao de Metricas - Modelos Individuais x Comite")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)          # respiro visual acima de 1.0
plt.xticks(rotation=20, ha="right")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../comparacao_metricas.png", dpi=120)
plt.show()

### 7.2) Matrizes de confusão — `matrizes_confusao.png`

**O que cada matriz de confusão mostra** (cada modelo tem a sua, todas plotadas em grade):

```
                          Predito
                       Nao        Sim
         Real  Nao  [  TN   |   FP  ]
               Sim  [  FN   |   TP  ]
```

- **TN (verdadeiro negativo):** asteroide não perigoso classificado como não perigoso. ✅
- **FP (falso positivo):** asteroide *seguro* classificado como perigoso. Custo: alarme falso.
- **FN (falso negativo):** asteroide *perigoso* classificado como seguro. **Erro mais crítico** no domínio — passar um PHA despercebido.
- **TP (verdadeiro positivo):** asteroide perigoso classificado como perigoso. ✅

**Como ler o gráfico:**
- Cor `Blues`: quanto mais escuro, maior a contagem naquela célula.
- O ideal é ter a **diagonal principal** (TN, TP) bem escura e os off-diagonals (FP, FN) bem claros.
- Modelos que apresentam **FN baixo** são os mais valiosos neste problema, mesmo que tenham um FP um pouco maior — é melhor disparar um alarme falso do que deixar passar um asteroide perigoso.

**Estrutura do código abaixo:**
- `n = len(resultados)` → 7 modelos.
- Grade `rows × cols`: definimos 4 colunas e calculamos as linhas com `ceil(n/cols)`.
- `axes.flat[n:]` desliga os subplots vazios (caso `rows*cols > n`) com `axis("off")`.

In [ ]:
# 7.2 Matrizes de confusao em grade (uma por modelo)
n = len(resultados)
cols = 4
rows = int(np.ceil(n / cols))    # arredonda para cima -> garante espaco p/ todos

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))

for ax, (nome, r) in zip(axes.flat, resultados.items()):
    ConfusionMatrixDisplay(
        r["cm"],
        display_labels=["Nao", "Sim"],
    ).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(nome)

# Desliga subplots sobrando (rows*cols pode ser > n)
for ax in axes.flat[n:]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("../matrizes_confusao.png", dpi=120)
plt.show()

## 8) Validação cruzada (5-fold estratificada) do Comitê Soft

Treinar e medir em um único split é frágil: a métrica pode estar "com sorte" daquele teste específico. Para checar **robustez**, rodamos uma **validação cruzada estratificada de 5 folds** sobre o conjunto balanceado completo:

- O dataset é dividido em 5 partes (mantendo a proporção 1:3 em cada).
- Treina em 4, testa em 1 — repete 5 vezes alternando o fold de teste.
- Reporta o **F1 médio** ± **desvio-padrão**. Desvio pequeno ⇒ modelo estável.

Observação importante: aqui re-escalamos `X_bal` com um `StandardScaler` novo, porque o `cross_val_score` precisa receber uma matriz já preparada (não estamos dentro de um `Pipeline`). Em um projeto maior, o ideal seria envelopar `scaler + ensemble_soft` num `Pipeline` para evitar mesmo o pequeno vazamento entre folds — mas para os fins do trabalho A3 essa simplificação é aceitável.

In [ ]:
print("\n" + "=" * 60)
print("8) VALIDACAO CRUZADA (5-fold) DO COMITE SOFT")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scores = cross_val_score(
    ensemble_soft,
    StandardScaler().fit_transform(X_bal),  # padroniza o conjunto todo
    y_bal,
    cv=cv,
    scoring="f1",                            # F1 da classe positiva
)

print(f"F1 por fold : {np.round(scores, 4)}")
print(f"F1 medio    : {scores.mean():.4f}  (+/- {scores.std():.4f})")

## 9) Top 5 asteroides com maior risco de colisão

Esta seção é mais **expositiva** do que de modelagem: dentre os asteroides já classificados oficialmente como `PHA = Y`, listamos os **5 com menor MOID** (*Minimum Orbit Intersection Distance*) — ou seja, os que chegam geometricamente mais perto da órbita da Terra.

- **MOID em UA** → distância mínima de aproximação entre as órbitas, em unidades astronômicas.
- **MOID em km** → converte para escala humana (1 UA = 149.597.870,7 km).
- **Período em anos** → `per` no SBDB vem em dias, dividimos por 365,25 para anos.
- Mostramos também `H` (magnitude → proxy de tamanho), `e` (excentricidade), `a` (semieixo), `i` (inclinação) e `class` (família orbital).

⚠️ Esse ranking é **geométrico**, não usa o modelo treinado. Serve para responder "quais asteroides reais merecem mais atenção?" — uma camada de contexto que vai além das métricas estatísticas.

In [ ]:
print("\n" + "=" * 60)
print("9) TOP 5 ASTEROIDES COM MAIOR RISCO DE COLISAO")
print("=" * 60)

AU_KM = 149_597_870.7   # 1 UA em quilometros (conversao p/ escala humana)

top5 = (
    df[df["pha"] == "Y"]                      # so PHAs oficiais
    .dropna(subset=["moid"])                  # precisa ter MOID definido
    .nsmallest(5, "moid")                     # 5 menores MOIDs => maior risco
    .loc[:, ["full_name", "H", "moid", "per", "e", "a", "i", "class"]]
    .copy()
)

top5["moid_km"]   = (top5["moid"] * AU_KM).round(0)     # MOID em km
top5["per_anos"]  = (top5["per"] / 365.25).round(2)     # periodo em anos
top5["full_name"] = top5["full_name"].str.strip()

print(top5[["full_name", "H", "moid", "moid_km",
            "per_anos", "e", "a", "i", "class"]].to_string(index=False))

print("\nArquivos gerados:")
print("  - comparacao_metricas.png")
print("  - matrizes_confusao.png")
print("\nFim.")

## Conclusão e leitura dos resultados

Recapitulando o que o notebook entrega, em ordem:

1. **Carregamento** → revela o desbalanceamento extremo (~0,18% de positivos).
2. **Pré-processamento** → encoding do alvo, seleção *consciente* de features (sem `moid`/`neo`, para evitar data leakage), imputação por mediana.
3. **Balanceamento 1:3** → torna o problema tratável sem jogar dados fora demais.
4. **Split + padronização** → estratificado, com `scaler` fitado só no treino.
5. **5 classificadores individuais** → KNN, NB, DT, SVM, MLP, cada um com viés indutivo distinto.
6. **2 comitês** → Soft (médio das probabilidades, principal) e Hard (maioria simples, comparativo).
7. **Gráficos:**
   - `comparacao_metricas.png` → mostra Accuracy/Precision/Recall/F1 lado a lado para os 7 modelos.
   - `matrizes_confusao.png` → mostra TN/FP/FN/TP de cada modelo, evidenciando **onde** cada um erra.
8. **Validação cruzada** → checa que o F1 do comitê soft é **estável** em diferentes partições.
9. **Top 5 PHAs reais** → contextualiza o problema com asteroides concretos de menor MOID.

**No domínio:** o erro mais caro é o **falso negativo** (deixar passar um asteroide perigoso). Por isso, ao olhar os gráficos, vale priorizar modelos com **Recall alto na classe positiva** e **FN baixo** na matriz de confusão — mesmo que isso custe um pouco de Precision.